# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- Ollama running locally with `deepseek-r1:14b` pulled

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [2]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 76 nodes, 481 edges

Micro-CKG: 76 nodes, 481 edges


## 5.2 Create QA Agent (Ollama — deepseek-r1:14b)

The QA agent uses **deepseek-r1:14b** running locally via Ollama with a pruned graph context. Weak edges (|log2FC| < 0.25 and spatial correlation < 0.4) are removed to reduce prompt token size. No API key needed — runs entirely on-device.

In [3]:
import time

# Aggressively prune the graph to reduce LLM prompt token size,
# but always preserve drug/ortholog translational edges (they have no log2fc/spatial attrs)
_KEEP_EDGE_LABELS = {"drug_gene_association", "is_ortholog_of"}

optimized_graph = graph.copy()
edges_to_remove = [
    (u, v) for u, v, d in optimized_graph.edges(data=True)
    if d.get("label") not in _KEEP_EDGE_LABELS
    and abs(d.get("log2fc", 0)) < 0.25
    and d.get("spatial_correlation", 0) < 0.4
]
optimized_graph.remove_edges_from(edges_to_remove)

# Strip internal-only attributes so they don't appear in the LLM context
_INTERNAL_ATTRS = {"human_ortholog"}
for _, data in optimized_graph.nodes(data=True):
    for attr in _INTERNAL_ATTRS:
        data.pop(attr, None)

print(f"Original edges: {graph.number_of_edges()}")
print(f"Pruned edges for LLM: {optimized_graph.number_of_edges()}")

agent = create_qa_agent(optimized_graph, provider="ollama")
print("QA agent initialised (Ollama — deepseek-r1:14b).")

Original edges: 481
Pruned edges for LLM: 414


QA agent initialised (Ollama — deepseek-r1:14b).


## 5.3 Translational Interview Queries

Four questions designed to demonstrate the end-to-end value of the Micro-CKG for a Takeda Pharmaceutical interview — each exercises a different translational capability of the pipeline:

1. **Translational drug discovery** — which FDA/clinical compounds in ChEMBL target the human orthologs of Stabl-selected AD biomarkers?
2. **Spatial pathway analysis** — which spatially autocorrelated genes are linked to the hippocampus and what do the cell-type associations imply for AD neurodegeneration?
3. **Cluster-specific drug targets** — within Leiden Cluster 3, which Stabl DE genes have human orthologs with PhaseIII+ drug matches?
4. **PageRank hub gene druggability** — which gene is the highest-connectivity hub connecting cell types and regions, and does its human ortholog have an FDA-approved drug?

In [4]:
print("[Query 1] Translational drug discovery...")
print(query_graph(
    agent,
    "Which FDA-approved or clinical-stage drugs in the Micro-CKG target the human orthologs "
    "of Stabl-selected AD biomarkers? For each drug-target pair, cite the full translational "
    "evidence path: mouse gene → human ortholog → drug compound, and include mechanism of "
    "action and clinical phase.",
))

[Query 1] Translational drug discovery...
  Querying agent: Which FDA-approved or clinical-stage drugs in the Micro-CKG target the human ort...


Based on the analysis of the provided mouse genes and their human orthologs, there are no FDA-approved drugs targeting the human orthologs of the Stabl-selected AD biomarkers. However, there are some compounds in clinical trials that may be relevant. Here is a summary of the findings:

### Key Findings:
1. **SPARC (Sparc)**: No FDA-approved drugs targeting SPARC directly. Preclinical studies may be ongoing.
2. **PRKCG (Prkcg)**: Some kinase inhibitors in preclinical stages, but no approved drugs.
3. **SGK1 (Sgk1)**: Inhibitors are under preclinical research.
4. **TSC22D4 (Tsc22d4)**: No specific drugs identified in clinical trials.

### Conclusion:
There are no FDA-approved drugs targeting the human orthologs of the Stabl-selected AD biomarkers listed. While some compounds may be in clinical trials, further research is needed to identify specific drugs and their mechanisms of action.


In [5]:
print("\n[Query 2] Spatial pathway + anatomy...")
print(query_graph(
    agent,
    "Which spatially autocorrelated Stabl genes are associated with the hippocampus in the "
    "Micro-CKG? Identify the connecting cell-type nodes and describe what their expression "
    "profiles suggest about AD-driven neurodegeneration in this anatomical region.",
))


[Query 2] Spatial pathway + anatomy...
  Querying agent: Which spatially autocorrelated Stabl genes are associated with the hippocampus i...


**Answer:**

In the context of the Micro-CKG (Mouse Cerebellum, Hippocampus, and Olfactory Bulb Cell Atlas), the spatially autocorrelated Stabl genes associated with the hippocampus are Trf, Sparc, and Tsc22d4. These genes are connected to the hippocampus cell-type node, specifically Cluster_13_Hippocampus.

1. **Trf (gene:Trf):** Orthologous to TF in humans, Trf is involved in transcriptional regulation and stress responses. Its expression in the hippocampus suggests a role in the brain's response to neurodegenerative stress, potentially contributing to inflammatory processes in Alzheimer's Disease (AD).

2. **Sparc (gene:Sparc):** Known for its role in extracellular matrix regulation and inflammation, Sparc may be involved in the response to amyloid-beta plaques or tau tangles in AD, indicating a repair or pathological role.

3. **Tsc22d4 (gene:Tsc22d4):** Involved in cell cycle regulation and apoptosis, Tsc22d4's expression suggests a role in cell survival or death processes, which 

In [6]:
print("\n[Query 3] Cluster-specific drug targets...")
# Note: Leiden Cluster 3 is assigned deterministically (seeded), but cluster
# labels may shift if the preprocessing or clustering parameters change.
print(query_graph(
    agent,
    "Within Leiden Cluster 3, which differentially expressed Stabl genes have human orthologs "
    "with matching ChEMBL drug entries in the Micro-CKG? Trace the evidence path from log2FC "
    "and p_adj through the ortholog mapping to the drug candidate, and flag any drugs at "
    "max_phase >= 3.",
))


[Query 3] Cluster-specific drug targets...
  Querying agent: Within Leiden Cluster 3, which differentially expressed Stabl genes have human o...


To address the question, we need to follow a structured approach to identify differentially expressed genes in Leiden Cluster 3, map them to their human orthologs, and check for associated drugs in the ChEMBL database via the Micro-CKG. Here's the step-by-step methodology:

### 1. Identify Differentially Expressed Genes in Leiden Cluster 3
- **Objective**: Locate genes within Leiden Cluster 3 that show significant differential expression, indicated by |log2FC| >1 and p_adj <0.05.
- **Action**: Review the provided data for genes in Cluster 3. If Cluster 3 isn't listed, consider hypothetical genes or refer to a different dataset.

### 2. Map Mouse Genes to Human Orthologs
- **Objective**: Convert identified mouse genes to their human orthologs using the provided ortholog mapping.
- **Example**: Mouse gene "Tsc22d4" maps to human gene "TSC22D4."

### 3. Query the Micro-CKG for ChEMBL Drug Entries
- **Objective**: For each human ortholog, check the Micro-CKG database for associated ChEMBL 

In [7]:
print("\n[Query 4] PageRank hub gene druggability...")
print(query_graph(
    agent,
    "Based on the topology of the Micro-CKG, which gene node has the highest connectivity to "
    "both cell types and anatomical regions? Is there an FDA-approved drug in the graph that "
    "targets its human ortholog? Provide the complete evidence chain, including PageRank score "
    "if available, ortholog mapping, and drug mechanism of action.",
))


[Query 4] PageRank hub gene druggability...
  Querying agent: Based on the topology of the Micro-CKG, which gene node has the highest connecti...


The gene node with the highest connectivity in the Micro-CKG is **Trf** (mouse gene), which is orthologous to the human gene **TF**. Trf has connections to five anatomical regions (Cerebellum, Cortex, Hippocampus, Thalamus, White Matter) and multiple cell types through its human ortholog TF. 

Regarding FDA-approved drugs, the provided graph does not explicitly mention any drugs targeting TF. Therefore, based on the given data, there is no FDA-approved drug identified that targets the human ortholog of Trf.

**Evidence Chain:**
1. **Connectivity Analysis:**
   - **Trf** is connected to five anatomical regions and multiple cell types via its human ortholog TF, indicating high connectivity.
   
2. **Ortholog Mapping:**
   - **Trf (mouse)** is the ortholog of **TF (human)**.

3. **Drug Information:**
   - No FDA-approved drugs targeting TF are mentioned in the provided graph.

**Conclusion:**
Trf has the highest connectivity in the Micro-CKG, but no FDA-approved drug targeting its human o